# FBNeoRageX 아바타 생성기 v3 (Google Colab)

**v2 에서 일관성이 깨진 이유와 v3 의 해결 방식**
- v2: 몸을 AI 가 자유롭게 그려서 자세·크기·배경이 매번 달랐고, 그 위에 옷/모자를 얹으니 어긋났습니다.
- v3: **고정된 마네킹 도안**(머리·몸통·팔·다리 위치가 항상 같음)에서 시작합니다. AI 는 그 도안 위에 그림을 입히기만 하므로 **모든 부품의 위치가 항상 같습니다.**
- 몸은 후보를 여러 장 뽑아 **직접 마음에 드는 것을 고릅니다** (남/여 각 1개).
- 얼굴(머리카락 포함) → 상의 → 모자는 그 몸 위에 같은 위치로 그려서 레이어로 뽑습니다.

**구성 (사용자 지정)**: 몸 2종(남/여) · 얼굴 남 10 + 여 10 · 상의 남 10 + 여 10 · 모자 10 (남녀 공용)

**사용 순서**
1. `런타임 → 런타임 유형 변경 → T4 GPU`
2. 1번 셀 실행 → `OK` 확인
3. 2~4번 셀 실행 (설정, 모델, 도우미)
4. **5번 셀**: 몸 후보를 뽑아 번호가 붙은 그림을 봅니다. 마음에 드는 번호를 **6번 셀의 `PICK`** 에 적습니다.
5. 6번 셀부터 끝까지 실행 → 마지막에 `avatars.zip` 이 내려옵니다.
6. 풀어서 `assets/avatars/` 에 넣습니다.

마음에 안 드는 부품은 그 셀만 `SEED` 를 바꿔 다시 돌리면 됩니다.

In [ ]:
#@title 1) 준비 (설치 + 점검)
!pip install -q diffusers accelerate
import importlib
problems = []
for m in ('numpy', 'scipy', 'PIL', 'torch', 'transformers', 'diffusers', 'accelerate'):
    try:
        mod = importlib.import_module(m); print(f'{m:13s}', getattr(mod, '__version__', '?'))
    except Exception as e:
        problems.append((m, e)); print(f'{m:13s} 실패 ->', repr(e)[:120])
try:
    import torch
    if not torch.cuda.is_available(): problems.append(('GPU', '런타임 유형이 GPU 가 아닙니다'))
    else: print('GPU          ', torch.cuda.get_device_name(0))
    from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionInpaintPipeline
except Exception as e:
    problems.append(('import', e))
if problems:
    print(); print('!! 문제:', problems)
    print('   -> [런타임 → 세션 다시 시작] 후 1번 셀을 다시 실행하세요.')
else:
    print(); print('OK - 다음 셀로 진행하세요.')

In [ ]:
#@title 2) 설정 — 부품 목록 (이름·프롬프트·해금 레벨)
SEED = 1234
STYLE = ("cute chibi arcade fighting game character, clean thick black outline, flat cel shading, "
         "vivid colors, front view, full body, isolated on plain pure white background, no shadow")
NEGATIVE = ("background, pattern, scenery, floor, shadow, multiple characters, text, watermark, blurry, "
            "low quality, cropped, side view, back view, extra limbs")

# 해금 레벨: 10개 부품에 순서대로 적용 (원하면 고치세요)
LEVELS = [1, 1, 5, 10, 20, 30, 40, 50, 70, 90]

BODY = {   # 몸: 헐벗지 않게 기본 속옷 차림, 머리카락 없음(얼굴 부품이 머리를 덮습니다)
    'm': ("young man, bald head, blank simple face, muscular slim build, plain gray sleeveless undershirt, gray shorts, bare arms", ("남자", "MALE")),
    'f': ("young woman, bald head, blank simple face, slim athletic build, plain gray sleeveless undershirt, gray shorts, bare arms", ("여자", "FEMALE")),
}
# 얼굴(머리 모양 포함): (id, 한글, 영문, 프롬프트)
FACES = {
 'm': [
  ("m_face01", "검은 단발",   "BLACK SHORT",  "short black hair, calm eyes"),
  ("m_face02", "금발 스파이크", "BLOND SPIKY",  "spiky blond hair, confident smile"),
  ("m_face03", "갈색 가르마",  "BROWN PART",   "brown side-parted hair, friendly smile"),
  ("m_face04", "빨강 머리",    "RED HAIR",     "messy red hair, determined eyes"),
  ("m_face05", "긴 묶음 머리", "LONG TIED",    "long black hair tied back in a ponytail, sharp eyes"),
  ("m_face06", "모히칸",      "MOHAWK",       "green mohawk hair, grin"),
  ("m_face07", "곱슬머리",    "CURLY",        "curly dark brown hair, cheerful"),
  ("m_face08", "흰 머리",     "WHITE HAIR",   "white spiky hair, cool expression"),
  ("m_face09", "대머리 수염",  "BALD BEARD",   "bald with a thick black beard, stern face"),
  ("m_face10", "파란 머리",    "BLUE HAIR",    "blue swept-back hair, smirk"),
 ],
 'f': [
  ("f_face01", "포니테일",    "PONYTAIL",     "black ponytail hair, bright eyes"),
  ("f_face02", "양갈래",      "TWIN TAILS",   "orange twin tail hair, cheerful smile"),
  ("f_face03", "긴 생머리",   "LONG STRAIGHT","long straight black hair, calm eyes"),
  ("f_face04", "분홍 단발",   "PINK BOB",     "pink bob cut hair, cute smile"),
  ("f_face05", "땋은 머리",   "BRAID",        "long brown braided hair, gentle eyes"),
  ("f_face06", "파란 숏컷",   "BLUE SHORT",   "short blue hair, confident smile"),
  ("f_face07", "올림머리",    "BUN",          "dark hair in a high bun, sharp eyes"),
  ("f_face08", "빨강 긴머리",  "LONG RED",     "long wavy red hair, fierce eyes"),
  ("f_face09", "금발 웨이브",  "BLOND WAVY",   "wavy blond hair, sweet smile"),
  ("f_face10", "흰 머리 앞머리","WHITE BANGS",  "white hair with straight bangs, cool expression"),
 ],
}
# 상의: (id, 한글, 영문, 프롬프트)  — 흰색 계열은 기본 속옷과 구분이 안 되니 피하세요
TOPS = {
 'm': [
  ("m_top01", "빨간 티셔츠", "RED TEE",       "a red t-shirt"),
  ("m_top02", "초록 티셔츠", "GREEN TEE",     "a green t-shirt"),
  ("m_top03", "파란 후드",   "BLUE HOODIE",   "a blue hooded sweatshirt"),
  ("m_top04", "가죽 재킷",   "LEATHER JACKET","a black leather jacket over a red shirt"),
  ("m_top05", "도복",       "GI",            "an orange karate gi top with a black belt"),
  ("m_top06", "정장",       "SUIT",          "a black business suit jacket with a red tie"),
  ("m_top07", "노랑 탱크톱", "YELLOW TANK",   "a yellow tank top"),
  ("m_top08", "청재킷",     "DENIM JACKET",  "a blue denim jacket"),
  ("m_top09", "격투 조끼",   "FIGHT VEST",    "a red fighting vest open at the chest with bandaged arms"),
  ("m_top10", "갑옷",       "ARMOR",         "a shiny silver armor chest plate"),
 ],
 'f': [
  ("f_top01", "분홍 티셔츠", "PINK TEE",      "a pink t-shirt"),
  ("f_top02", "하늘 티셔츠", "SKY TEE",       "a sky blue t-shirt"),
  ("f_top03", "빨간 후드",   "RED HOODIE",    "a red hooded sweatshirt"),
  ("f_top04", "가죽 재킷",   "LEATHER JACKET","a black leather jacket over a purple top"),
  ("f_top05", "도복",       "GI",            "a blue karate gi top with a black belt"),
  ("f_top06", "세일러복",    "SAILOR",        "a sailor school uniform top with a red ribbon"),
  ("f_top07", "노랑 탱크톱", "YELLOW TANK",   "a yellow tank top"),
  ("f_top08", "청재킷",     "DENIM JACKET",  "a blue denim jacket"),
  ("f_top09", "격투 조끼",   "FIGHT VEST",    "a green fighting vest with bandaged arms"),
  ("f_top10", "갑옷",       "ARMOR",         "a shiny golden armor chest plate"),
 ],
}
# 모자 10개 (남녀 공용)
HATS = [
  ("hat01", "빨간 야구모자", "RED CAP",      "a red baseball cap"),
  ("hat02", "머리띠",       "HEADBAND",     "a red headband tied around the forehead"),
  ("hat03", "비니",         "BEANIE",       "an orange knit beanie"),
  ("hat04", "왕관",         "CROWN",        "a golden king crown"),
  ("hat05", "밀짚모자",      "STRAW HAT",    "a yellow straw hat"),
  ("hat06", "마법사 모자",   "WIZARD HAT",   "a purple pointed wizard hat"),
  ("hat07", "헬멧",         "HELMET",       "a silver knight helmet"),
  ("hat08", "고양이 귀",     "CAT EARS",     "a headband with cat ears"),
  ("hat09", "중절모",       "FEDORA",       "a brown fedora hat"),
  ("hat10", "해적 모자",     "PIRATE HAT",   "a black pirate hat with a skull"),
]

In [ ]:
#@title 3) 모델 불러오기 (처음에는 몇 분 걸립니다)
import os, json, zipfile
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFilter, ImageChops
from scipy import ndimage
from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionInpaintPipeline

assert torch.cuda.is_available(), '런타임 유형을 T4 GPU 로 바꾸세요'
dev = 'cuda'
kw = dict(torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False)
img2img = StableDiffusionImg2ImgPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-v1-5', **kw).to(dev)
inpaint = StableDiffusionInpaintPipeline.from_pretrained('stable-diffusion-v1-5/stable-diffusion-inpainting', **kw).to(dev)
for p in (img2img, inpaint):
    p.enable_attention_slicing(); p.set_progress_bar_config(disable=True)
W, H = 256, 320          # 최종 크기
GW, GH = 512, 640        # 작업 크기 (2배)
print('ready')

In [ ]:
#@title 4) 도안(마네킹)과 도우미 함수
# ── 고정 도안: 이 위치가 모든 부품의 기준입니다 (작업 크기 512x640) ──
HEAD_BOX  = (130, 10, 382, 262)     # 얼굴/머리카락이 그려지는 영역
TOP_BOX   = (100, 240, 412, 430)    # 상의가 그려지는 영역 (몸통 + 팔 윗부분)
HAT_BOX   = (120, 0, 392, 170)      # 모자가 그려지는 영역 (머리 위쪽)

def mannequin(gender):
    # 단색 회색 마네킹. AI 는 이 윤곽을 따라 그림을 입힙니다 (남: 어깨 넓게, 여: 좁게)
    im = Image.new('RGB', (GW, GH), (255, 255, 255)); d = ImageDraw.Draw(im)
    g = (150, 150, 150); sh = 100 if gender == 'm' else 88      # 어깨 반폭
    d.ellipse((151, 30, 361, 245), fill=g)                               # 머리
    d.rectangle((236, 235, 276, 265), fill=g)                            # 목
    d.rounded_rectangle((256 - sh, 262, 256 + sh, 400), 28, fill=g)      # 몸통
    d.rounded_rectangle((256 - sh - 38, 268, 256 - sh + 6, 420), 20, fill=g)   # 왼팔
    d.rounded_rectangle((256 + sh - 6, 268, 256 + sh + 38, 420), 20, fill=g)   # 오른팔
    d.rounded_rectangle((256 - 70, 395, 256 - 4, 590), 22, fill=g)       # 왼다리
    d.rounded_rectangle((256 + 4, 395, 256 + 70, 590), 22, fill=g)       # 오른다리
    d.ellipse((256 - 88, 570, 256 - 2, 625), fill=g); d.ellipse((256 + 2, 570, 256 + 88, 625), fill=g)   # 발
    return im

def cutout_white(img, tol=30):
    # 가장자리와 이어진 흰색만 배경으로 보고 투명하게 (캐릭터 안쪽 흰색은 남음)
    a = np.asarray(img.convert('RGB')).astype(np.int16)
    near_white = (a.min(axis=2) >= 255 - tol)
    lab, n = ndimage.label(near_white)
    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))) - {0}
    bg = np.isin(lab, list(border)) if border else np.zeros_like(near_white)
    alpha = Image.fromarray(np.where(bg, 0, 255).astype(np.uint8)).filter(ImageFilter.MinFilter(3)).filter(ImageFilter.GaussianBlur(0.8))
    out = img.convert('RGBA'); out.putalpha(alpha); return out

def over_white(rgba):
    bg = Image.new('RGB', rgba.size, (255, 255, 255)); bg.paste(rgba, mask=rgba.getchannel('A')); return bg

def box_mask(box, blur=6):
    m = Image.new('L', (GW, GH), 0); m.paste(255, box); return m.filter(ImageFilter.GaussianBlur(blur))

def clean_bg(img, tol=40):
    # 배경이 완전히 흰색이 아니면(희미한 무늬) 가장자리와 이어진 부분을 흰색으로 덮는다
    return over_white(cutout_white(img, tol))

def solid_alpha(cond, box, grow=2):
    # 조건(바뀐 곳/배경 아님) → 구멍을 메운 단단한 알파 (box 안쪽만)
    cond = ndimage.binary_dilation(cond, iterations=grow)
    cond = ndimage.binary_fill_holes(cond)
    cond = ndimage.binary_closing(cond, iterations=2)
    keep = np.zeros_like(cond); x0, y0, x1, y1 = box; keep[y0:y1, x0:x1] = True
    a = Image.fromarray((cond & keep).astype(np.uint8) * 255)
    return a.filter(ImageFilter.MinFilter(3)).filter(ImageFilter.GaussianBlur(0.9))

def to_layer(rgb, alpha):
    out = rgb.convert('RGBA'); out.putalpha(alpha); return out.resize((W, H), Image.LANCZOS)

def run_inpaint(base_rgb, box, prompt, seed, strength=1.0):
    mask = box_mask(box)
    g = torch.Generator(dev).manual_seed(seed)
    out = inpaint(prompt=f"{prompt}, {STYLE}", negative_prompt=NEGATIVE, image=base_rgb, mask_image=mask,
                  width=GW, height=GH, num_inference_steps=30, guidance_scale=7.5, strength=strength, generator=g).images[0].convert('RGB')
    out = clean_bg(out)
    hard = box_mask(box, blur=1).point(lambda v: 255 if v > 128 else 0)
    return Image.composite(out, base_rgb, hard)        # 박스 바깥은 원본 그대로 (번짐 방지)

def show_row(images, bgcolor=(255, 255, 255), labels=None):
    row = []
    for im in images:
        im = im.convert('RGBA'); bg = Image.new('RGB', im.size, bgcolor); bg.paste(im, mask=im.getchannel('A')); row.append(np.asarray(bg))
    display(Image.fromarray(np.hstack(row)))

# 도안 미리보기
show_row([mannequin('m').convert('RGBA'), mannequin('f').convert('RGBA')])

In [ ]:
#@title 5) 몸 후보 뽑기 — 번호를 보고 마음에 드는 것을 6번 셀에서 고르세요
N_CAND = 6
cands = {}
for gnd in ('m', 'f'):
    tpl = mannequin(gnd); cands[gnd] = []
    for k in range(N_CAND):
        g = torch.Generator(dev).manual_seed(SEED + (0 if gnd == 'm' else 1000) + k)
        im = img2img(prompt=f"{BODY[gnd][0]}, {STYLE}", negative_prompt=NEGATIVE, image=tpl, strength=0.78,
                     num_inference_steps=40, guidance_scale=8.0, generator=g).images[0].convert('RGB')
        cands[gnd].append(clean_bg(im))
    print('남자' if gnd == 'm' else '여자', '후보 번호: 0 ~', N_CAND - 1)
    show_row([c.resize((GW // 2, GH // 2)).convert('RGBA') for c in cands[gnd]])

In [ ]:
#@title 6) 몸 고르기 — 위 그림의 번호를 적고 실행
PICK = {'m': 0, 'f': 0}      # ← 남자/여자 번호를 바꾸세요 (마음에 드는 게 없으면 SEED 를 바꿔 5번 셀 다시)
os.makedirs('avatars/body', exist_ok=True)
base = {}                    # 작업 크기 RGB (부품 생성의 기준)
body_ids = {'m': 'body_m', 'f': 'body_f'}
for gnd in ('m', 'f'):
    base[gnd] = cands[gnd][PICK[gnd]]
    cutout_white(base[gnd]).resize((W, H), Image.LANCZOS).save(f'avatars/body/{body_ids[gnd]}.png')
show_row([Image.open(f'avatars/body/{body_ids[g]}.png') for g in ('m', 'f')], bgcolor=(40, 90, 200))

In [ ]:
#@title 7) 얼굴(머리 모양) 만들기 — 남 10 + 여 10
os.makedirs('avatars/face', exist_ok=True)
face_rgb = {}                # 모자 생성에서 기준 얼굴로 씁니다
for gnd in ('m', 'f'):
    imgs = []
    for i, (fid, ko, en, pr) in enumerate(FACES[gnd]):
        ed = run_inpaint(base[gnd], HEAD_BOX, f"{pr}, head only", SEED + 300 + i + (0 if gnd == 'm' else 50))
        nonbg = np.asarray(cutout_white(ed).getchannel('A')) > 128
        layer = to_layer(ed, solid_alpha(nonbg, HEAD_BOX, grow=1))
        layer.save(f'avatars/face/{fid}.png'); imgs.append(layer)
        if i == 0: face_rgb[gnd] = ed
    show_row(imgs, bgcolor=(40, 90, 200))

In [ ]:
#@title 8) 상의 만들기 — 남 10 + 여 10
os.makedirs('avatars/top', exist_ok=True)
for gnd in ('m', 'f'):
    imgs = []; base_arr = np.asarray(base[gnd]).astype(np.int16)
    for i, (tid, ko, en, pr) in enumerate(TOPS[gnd]):
        ed = run_inpaint(base[gnd], TOP_BOX, f"wearing {pr}, torso and arms only", SEED + 600 + i + (0 if gnd == 'm' else 50))
        diff = np.abs(np.asarray(ed).astype(np.int16) - base_arr).max(axis=2) > 30
        layer = to_layer(ed, solid_alpha(diff, TOP_BOX, grow=2))
        layer.save(f'avatars/top/{tid}.png'); imgs.append(layer)
    show_row(imgs, bgcolor=(40, 90, 200))

In [ ]:
#@title 9) 모자 만들기 — 10개 (남녀 공용, 첫 번째 얼굴 위에 씌워서 뽑습니다)
os.makedirs('avatars/hat', exist_ok=True)
ref = face_rgb['m']                       # 기준: 남자 첫 번째 얼굴 (여자 얼굴과 머리 위치가 같은 도안이라 호환됩니다)
ref_arr = np.asarray(ref).astype(np.int16)
imgs = []
for i, (hid, ko, en, pr) in enumerate(HATS):
    ed = run_inpaint(ref, HAT_BOX, f"wearing {pr} on the head", SEED + 900 + i)
    diff = np.abs(np.asarray(ed).astype(np.int16) - ref_arr).max(axis=2) > 30
    layer = to_layer(ed, solid_alpha(diff, HAT_BOX, grow=3))
    layer.save(f'avatars/hat/{hid}.png'); imgs.append(layer)
Image.new('RGBA', (W, H), (0, 0, 0, 0)).save('avatars/hat/none.png')
show_row(imgs, bgcolor=(40, 90, 200))

In [ ]:
#@title 10) 합쳐서 미리보기 (몸 + 얼굴 + 상의 + 모자)
import random
def composite(gnd, f, t, h):
    c = Image.open(f'avatars/body/{body_ids[gnd]}.png').convert('RGBA')
    for slot, i in (('face', FACES[gnd][f][0]), ('top', TOPS[gnd][t][0]), ('hat', 'none' if h < 0 else HATS[h][0])):
        c.alpha_composite(Image.open(f'avatars/{slot}/{i}.png').convert('RGBA'))
    return c
random.seed(SEED)
for gnd in ('m', 'f'):
    show_row([composite(gnd, random.randrange(10), random.randrange(10), random.choice([-1] + list(range(10)))) for _ in range(5)], bgcolor=(40, 90, 200))

In [ ]:
#@title 11) catalog.json 만들고 avatars.zip 내려받기
def lv(i): return LEVELS[i]
cat = []
for gnd in ('m', 'f'):
    cat.append({"slot": "body", "id": body_ids[gnd], "ko": BODY[gnd][1][0], "en": BODY[gnd][1][1], "level": 1, "gender": gnd})
    for i, (iid, ko, en, pr) in enumerate(FACES[gnd]): cat.append({"slot": "face", "id": iid, "ko": ko, "en": en, "level": lv(i), "gender": gnd})
    for i, (iid, ko, en, pr) in enumerate(TOPS[gnd]):  cat.append({"slot": "top",  "id": iid, "ko": ko, "en": en, "level": lv(i), "gender": gnd})
cat.append({"slot": "hat", "id": "none", "ko": "없음", "en": "NONE", "level": 1, "gender": ""})
for i, (iid, ko, en, pr) in enumerate(HATS): cat.append({"slot": "hat", "id": iid, "ko": ko, "en": en, "level": lv(i), "gender": ""})
json.dump(cat, open('avatars/catalog.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
with zipfile.ZipFile('avatars.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk('avatars'):
        for f in fs: z.write(os.path.join(root, f))
try:
    from google.colab import files
    files.download('avatars.zip')
except Exception as e:
    print('avatars.zip 이 왼쪽 파일 창에 생겼습니다. 직접 다운로드하세요.', e)

## 결과 폴더
```
avatars/
  catalog.json                (slot: body/face/top/hat, gender: m/f/"")
  body/body_m.png body_f.png
  face/m_face01.png ... f_face10.png
  top/m_top01.png ... f_top10.png
  hat/hat01.png ... hat10.png, none.png
```
**프로그램 쪽**: 얼굴(face) 슬롯과 남/여 구분(gender)은 FBNeoRageX 새 빌드에서 지원합니다. 이 `avatars` 폴더를 `assets/` 안에 넣으세요.